In [ ]:
# 1. Install all required libraries silently
!pip install ultralytics easyocr opencv-python-headless pandas kaggle -q

import os
import shutil
import torch
from google.colab import drive, userdata

# 2. Verify GPU Status
if torch.cuda.is_available():
    print(f"✅ GPU is active: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU is not active. Go to Runtime > Change runtime type > Hardware accelerator > GPU")

# 3. Mount Google Drive
drive.mount('/content/drive')

# 4. Build Shared Project Architecture on Drive
PROJECT_ROOT = '/content/drive/MyDrive/MultiCamera_Vehicle_ReIdentification'
folders = ['models', 'notebooks', 'src', 'outputs', 'checkpoints', 'yolo_runs']

print("\nEnsuring project architecture exists...")
os.makedirs(PROJECT_ROOT, exist_ok=True)
for folder in folders:
    os.makedirs(os.path.join(PROJECT_ROOT, folder), exist_ok=True)

print(f"✅ Shared project folder ready at: {PROJECT_ROOT}")

In [ ]:
# 1. Define Local Dataset Directory
DATASET_DIR = '/content/dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

# 2. Authenticate Kaggle via Colab Secrets
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
    os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
    print("✅ Kaggle credentials loaded securely.")
except Exception:
    print("❌ Error: Missing Kaggle secrets. Set 'KAGGLE_USERNAME' and 'KAGGLE_KEY' in the Colab sidebar.")

# 3. Download and Unzip
print("⬇️ Downloading 'Large License Plate Dataset'...")
!kaggle datasets download -d fareselmenshawii/large-license-plate-dataset -p /content

print("📦 Unzipping dataset...")
# -q makes it quiet, -o overwrites existing files to prevent duplicate folder errors
!unzip -q -o /content/large-license-plate-dataset.zip -d {DATASET_DIR}

print("✅ Dataset downloaded and unzipped successfully to local Colab storage.")

In [ ]:
# 1. Create the YAML configuration with absolute paths
yaml_content = f"""
path: {DATASET_DIR}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['license_plate']
"""

local_yaml_path = os.path.join(DATASET_DIR, 'data.yaml')

with open(local_yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✅ Local config created at: {local_yaml_path}")

# 2. Backup the configuration to your shared Google Drive
drive_yaml_path = os.path.join(PROJECT_ROOT, 'data.yaml')
shutil.copy(local_yaml_path, drive_yaml_path)

print(f"✅ Config backed up to Drive: {drive_yaml_path}")
print("\n🎉 Setup Complete! You are now ready to run your training scripts.")